# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnanwubeikenna-prog/ikenna-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

1 row = 1 pseudonymized content item (`dim_content`, 519,606 rows) across 104 clients, with the warehouse tracking performance over ~17 months (2025-01-27 to 2026-06-30).

In [ ]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
con.execute("CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')")

display(con.sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id) AS unique_content, COUNT(DISTINCT client_hash_id) AS unique_clients FROM dim_content").df())

## 2. Fields: feature / label / context / excluded

- Features (X): Pre-cutoff historical content metrics (e.g. content_type, search_volume, word_count, backlinks).
- Label (y): is_declining_label (post-cutoff binary decline).
- Context: content_hash_id, client_hash_id (for joins and grouped client splits).
- Excluded: trend_pct and trend_direction (target leakage, as label is derived from them).

In [ ]:
display(con.sql("DESCRIBE dim_content").df())

## 3. Verify it with queries (grain, counts, missing values, windows)

Summarize that the grain holds with 0 duplicates on content_hash_id and missingness varies by content_type.

In [ ]:
# Grain check (must return 0 duplicate rows)
display(con.sql("SELECT content_hash_id, COUNT(*) AS cnt FROM dim_content GROUP BY content_hash_id HAVING COUNT(*) > 1 LIMIT 5").df())

# Missingness check by content_type
display(con.sql("""
    SELECT
        content_type,
        COUNT(*) AS n,
        AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0.0 END) AS pct_missing_word_count
    FROM dim_content
    GROUP BY content_type
""").df())

## 4. Data limits

Panel limits: varying client history starts, unmeasured external traffic, and content status flags.

In [7]:
# Inspect status flags and unindexed/zero search volume edge cases
display(con.sql("""
    SELECT
        COUNT(*) AS total_items,
        COUNT(CASE WHEN is_published = FALSE THEN 1 END) AS unpublished_items,
        COUNT(CASE WHEN is_deleted = TRUE THEN 1 END) AS deleted_items,
        COUNT(CASE WHEN search_volume = 0 OR search_volume IS NULL THEN 1 END) AS zero_or_null_search_vol
    FROM dim_content
""").df())

,total_items,unpublished_items,deleted_items,zero_or_null_search_vol
0,519606,108066,101559,306253


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.